In [1]:
import librosa
import torch
import torchaudio

import matplotlib.pyplot as plt
import numpy as np
from torchvision import transforms
import torch.nn.functional as F

import pandas as pd
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import h5py

from transformers import Wav2Vec2FeatureExtractor
from transformers import AutoModel

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [3]:
df = pd.read_feather('/kaggle/input/datasets/alexandra841/tracks-features/df_features_for_model', columns=['track_id', 'path', 'major_mood_class'])

In [4]:
paths = df['path'].tolist()

In [5]:
df['major_mood_class'].unique()

array(['calm', 'sad_dark', 'positive', 'energetic', 'deep_emotional',
       'romantic'], dtype=object)

In [6]:
mood_dict = dict(zip(df['major_mood_class'].unique(), range(len(df['major_mood_class'].unique()))))

In [7]:
mood_dict

{'calm': 0,
 'sad_dark': 1,
 'positive': 2,
 'energetic': 3,
 'deep_emotional': 4,
 'romantic': 5}

In [8]:
df['major_mood_class_int'] = df['major_mood_class'].map(mood_dict)

In [9]:
model = AutoModel.from_pretrained("m-a-p/MERT-v1-95M", trust_remote_code=True)
processor = Wav2Vec2FeatureExtractor.from_pretrained("m-a-p/MERT-v1-95M",trust_remote_code=True)

config.json: 0.00B [00:00, ?B/s]

configuration_MERT.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/m-a-p/MERT-v1-95M:
- configuration_MERT.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_MERT.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/m-a-p/MERT-v1-95M:
- modeling_MERT.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


pytorch_model.bin:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/211 [00:00<?, ?B/s]

In [10]:
model = model.to(device)

In [11]:
target_prep_sr = processor.sampling_rate

In [12]:
target_prep_sr

24000

In [13]:
directory='/kaggle/input/datasets/alexandra841/mtg-tracks-data/data/'

In [14]:
class AudioDataSet(Dataset):

    def __init__(
        self, 
        paths, 
        df_meta,
        directory='/kaggle/input/datasets/alexandra841/mtg-tracks-data/data/',
        target_sr = 22050, 
        len_sec = 30
    ):

        self.paths = paths
        self.target_sample_rate = target_sr
        self.len_sec = len_sec
        self.directory=directory
        self.dict_track_info = df_meta.set_index("path").to_dict("index")

        
    def __len__(self):
        return len(self.paths)

    def extract_audio(self, path):
        
        waveform, sr = torchaudio.load(path)
        
        max_len = self.target_sample_rate  * self.len_sec

        if sr != self.target_sample_rate:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=self.target_sample_rate)
            waveform = resampler(waveform)
            sr = self.target_sample_rate 

        len_audio = waveform.shape[1]
            
        if len_audio < max_len:
            pad = max_len - len_audio
            waveform = F.pad(waveform, (0, pad))
        else:
            waveform = waveform[:, :max_len]

        return waveform

    def get_track_info(self, path):
        short_path = path.replace(self.directory, '')
        track_info = self.dict_track_info[short_path]
        
        class_mood = track_info['major_mood_class']
        class_mood_int = track_info['major_mood_class_int']
        track_id = track_info['track_id']

        return class_mood, class_mood_int, track_id, short_path
        

    def __getitem__(self, idx):

        path = self.paths[idx]
        waveform = self.extract_audio(path)
        class_mood, class_mood_int, track_id, short_path = self.get_track_info(path)
        

        return {
            'path': short_path, 
            'track_id': track_id, 
            'audio': waveform,
            'class_mood': class_mood,
            'class_mood_int': class_mood_int,
        }

In [15]:
dataset = AudioDataSet(
    (directory + df['path']).tolist(), 
    df_meta=df,
    target_sr=target_prep_sr, 
    len_sec=30)

loader = DataLoader(dataset, batch_size=8, num_workers=2)

In [16]:
N = len(dataset)
str_dt = h5py.string_dtype(encoding="utf-8")

In [17]:
model.eval()

MERTModel(
  (feature_extractor): HubertFeatureEncoder(
    (conv_layers): ModuleList(
      (0): HubertGroupNormConvLayer(
        (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,), bias=False)
        (activation): GELUActivation()
        (layer_norm): GroupNorm(512, 512, eps=1e-05, affine=True)
      )
      (1-4): 4 x HubertNoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
      (5-6): 2 x HubertNoLayerNormConvLayer(
        (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,), bias=False)
        (activation): GELUActivation()
      )
    )
  )
  (feature_projection): MERTFeatureProjection(
    (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
    (projection): Linear(in_features=512, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): HubertEncoder(
    (pos_conv_embed): HubertPositionalConvEmbedding(
      (conv): Parametr

In [18]:
def get_embeddings(waveform, model, processor):
    waveform = waveform.squeeze(1).cpu().numpy()
    #waveform_list = [x.numpy() for x in waveform]
    inputs = processor(waveform, sampling_rate=target_prep_sr, return_tensors="pt")
    inputs = {k:v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            outputs = model(**inputs)

    last_hidden_state = outputs.last_hidden_state

    mean_pool = last_hidden_state.mean(dim=1)
    max_pool = last_hidden_state.max(dim=1).values
    x_mlp = torch.cat([mean_pool, max_pool], dim=1)

    x_rnn = last_hidden_state.transpose(1,2)         
    x_rnn = F.adaptive_avg_pool1d(x_rnn, 256)
    x_rnn = x_rnn.transpose(1,2)

    return x_mlp, x_rnn

In [19]:
with h5py.File('/kaggle/working/features_audio_embeddings.h5', 'w') as f:
    f.create_dataset('path', shape=(N,), dtype=str_dt)
    f.create_dataset('track_id', shape=(N,), dtype='int64')
    f.create_dataset('class_mood_int', shape=(N,), dtype="int8")
    f.create_dataset('class_mood', shape=(N,), dtype=str_dt)
    f.create_dataset(
        'x_rnn',
        shape=(N,256,768),
        dtype="float16",
        compression="lzf"
    )
    f.create_dataset(
        'x_mlp',
        shape=(N,1536),
        dtype="float16",
        compression="lzf"
    )

    idx=0
    model.eval()

    for batch in tqdm(loader):

        x_mlp, x_rnn = get_embeddings(batch['audio'], model, processor)

        batch_size = len(batch['track_id'])

        f['path'][idx:idx+batch_size] = batch['path']
        f['track_id'][idx:idx+batch_size] = batch['track_id'].numpy()
        f['class_mood_int'][idx:idx+batch_size] = batch['class_mood_int'].numpy()
        f['class_mood'][idx:idx+batch_size] = batch['class_mood']

        f['x_rnn'][idx:idx+batch_size] = x_rnn.to(torch.float16).cpu().numpy()
        f['x_mlp'][idx:idx+batch_size] = x_mlp.to(torch.float16).cpu().numpy()

        idx += batch_size

100%|██████████| 1951/1951 [1:37:29<00:00,  3.00s/it]


In [20]:
# for batch in tqdm(loader):

#     x_mlp, x_rnn = get_embeddings(batch['audio'], model, processor)

#     batch_size = len(batch['track_id'])
#     break